# Doppl — Neo4j lineage-query spike (P6.11)

**THROWAWAY / EXPLORATORY.** This notebook is a timeboxed week-2 spike (ARCHITECTURE.md §10). It is **doc-only** and is **never a runtime dependency** — the demo path works with this notebook absent. It documents the four Cypher query shapes over the *derived* lineage export produced by `apps/api/src/projections/lineage-export.ts` (`lineageToExport(projection)`).

Safety: the export is **derived + read-only** (key safety rule #2). It is a pure transform of the P6.3 `LineageGraphProjection` (which is itself a projection of the authoritative `run_events` log). Neo4j is a **derived, non-authoritative** read model — the Postgres event log stays the single source of truth; nothing here writes back.

No live Neo4j instance runs in the build environment, so the Cypher cells below are **raw (documentation) cells**, not executed.

## Export shape (input to every query)

`lineageToExport(projection)` emits a storage-agnostic structure — no Neo4j driver, no Cypher in `apps/api`:

```json
{
  "nodes": [{ "id": "cand_win", "labels": ["Candidate"], "props": { "label": "Winner idea", "dataRef": "cand_win", "status": "selected", "metrics": { "fitness": 0.9, "novelty": 0.7 } } }],
  "edges": [{ "id": "agn_parent->agn_child", "source": "agn_parent", "target": "agn_child", "type": "reproduced", "props": {} }],
  "sequenceThrough": 8
}
```

Node labels are the PascalCase `LineageNodeType` (`Generation`/`Agenome`/`Candidate`/`Critic`/`Check`/`Score`); the `selected` winner is a `Candidate` with `props.status = 'selected'` (not a 7th label). Edge `type` is the relationship type (`reproduced` genealogy + structural `generated`/`reviewed_by`/`checked_by`/`scored_by`).

## Load the export into Neo4j (derived read model — idempotent MERGE)

```cypher
// :param export => (the lineageToExport(projection) JSON)
UNWIND $export.nodes AS n
CALL apoc.merge.node(n.labels, {id: n.id}, n.props) YIELD node
RETURN count(node);

UNWIND $export.edges AS e
MATCH (s {id: e.source}), (t {id: e.target})
CALL apoc.merge.relationship(s, e.type, {id: e.id}, e.props, t) YIELD rel
RETURN count(rel);
```

## Query shape 1 — ancestors of the winner

Walk the reproduction genealogy backward from the `selected` candidate to its founding agenomes.

```cypher
MATCH (w:Candidate {status: 'selected'})
MATCH (w)<-[:generated]-(a:Agenome)
MATCH path = (ancestor:Agenome)-[:reproduced*0..]->(a)
RETURN w.id AS winner, [x IN nodes(path) | x.id] AS lineage
ORDER BY length(path) DESC;
```

## Query shape 2 — parent contribution

For each reproduction edge, how much each parent contributed to a child's fitness/novelty (props ride on the nodes).

```cypher
MATCH (parent:Agenome)-[r:reproduced]->(child:Agenome)
OPTIONAL MATCH (child)-[:generated]->(c:Candidate)
RETURN parent.id AS parent, child.id AS child,
       collect({candidate: c.id, fitness: c.metrics.fitness, novelty: c.metrics.novelty}) AS offspring
ORDER BY parent;
```

## Query shape 3 — critic-kill patterns

Which critics' reviews precede a rejected/culled candidate — the `reviewed_by` edge plus the candidate's terminal status.

```cypher
MATCH (c:Candidate)-[:reviewed_by]->(critic:Critic)
WHERE c.status IN ['rejected', 'culled']
RETURN critic.label AS critic_mandate, c.status AS outcome, count(*) AS kills
ORDER BY kills DESC;
```

## Query shape 4 — lineage distance / diversity

Genealogical distance between candidates (hops over the reproduction tree) and novelty spread — a diversity proxy.

```cypher
MATCH (a:Candidate), (b:Candidate)
WHERE a.id < b.id
MATCH (a)<-[:generated]-(:Agenome)-[gp:reproduced*]-(:Agenome)-[:generated]->(b)
WITH a, b, min(length(gp)) AS genealogical_distance
RETURN a.id AS a, b.id AS b, genealogical_distance,
       abs(a.metrics.novelty - b.metrics.novelty) AS novelty_spread
ORDER BY genealogical_distance DESC, novelty_spread DESC;
```

---
_End of spike. Throwaway — not wired into the runtime; the export function (`lineageToExport`) is the durable artifact._